# 框架

```mermaid
%% WonderTrader WtDtServo 模块关系图（简化版）
graph TD
    subgraph Layer1["第1层: C API 接口层（对外）"]
        WtDtServo_H["WtDtServo.h"]:::apiStyle
        WtDtServo_CPP["WtDtServo.cpp"]:::apiStyle
        PorterDefs_H["PorterDefs.h<br/>回调函数定义"]:::defStyle
        WtDtServo_H --> WtDtServo_CPP
        WtDtServo_CPP -.-> PorterDefs_H
    end

    subgraph Layer2["第2层: 核心协调层"]
        WtDtRunner_H["WtDtRunner.h"]:::runnerStyle
        WtDtRunner_CPP["WtDtRunner.cpp"]:::runnerStyle
        WtDtRunner_H --> WtDtRunner_CPP
    end

    subgraph Layer3["第3层: 功能模块层"]
        WtDataManager_H["WtDataManager.h"]:::managerStyle
        WtDataManager_CPP["WtDataManager.cpp<br/>数据查询与K线生成"]:::managerStyle
        ParserAdapter_H["ParserAdapter.h"]:::adapterStyle
        ParserAdapter_CPP["ParserAdapter.cpp<br/>行情源适配"]:::adapterStyle
        WtDataManager_H --> WtDataManager_CPP
        ParserAdapter_H --> ParserAdapter_CPP
    end

    subgraph Layer4["第4层: 辅助工具层"]
        WtHelper_H["WtHelper.h"]:::utilStyle
        WtHelper_CPP["WtHelper.cpp<br/>路径管理"]:::utilStyle
        WtHelper_H --> WtHelper_CPP
    end

    %% 层间依赖关系
    WtDtServo_CPP -->|"调用单例"| WtDtRunner_CPP
    WtDtRunner_CPP -->|"初始化"| WtDataManager_CPP
    WtDtRunner_CPP -->|"管理"| ParserAdapter_CPP
    WtDtRunner_CPP -.->|"使用"| WtHelper_CPP
    ParserAdapter_CPP -.->|"使用"| WtHelper_CPP
    WtDtRunner_CPP -.->|"包含"| PorterDefs_H
    ParserAdapter_CPP -->|"回调"| WtDtRunner_CPP

    %% 数据流示意
    subgraph DataFlow["数据流向"]
        direction LR
        ExtApp["外部应用<br/>Python/C#"] -->|"1. 调用API"| WtDtServo_CPP
        WtDtServo_CPP -->|"2. 转发"| WtDtRunner_CPP
        WtDtRunner_CPP -->|"3. 查询"| WtDataManager_CPP
        ParserAdapter_CPP -->|"4. 推送Tick"| WtDtRunner_CPP
        WtDtRunner_CPP -->|"5. 回调"| WtDtServo_CPP
        WtDtServo_CPP -->|"6. 推送"| ExtApp
    end

    %% 样式定义
    classDef apiStyle fill:#c9daf8,stroke:#333,stroke-width:2px
    classDef runnerStyle fill:#d9ead3,stroke:#333,stroke-width:2px
    classDef managerStyle fill:#f4cccc,stroke:#333,stroke-width:2px
    classDef adapterStyle fill:#fce5cd,stroke:#333,stroke-width:2px
    classDef defStyle fill:#fff2cc,stroke:#333,stroke-width:2px,stroke-dasharray: 5 5
    classDef utilStyle fill:#d9d2e9,stroke:#333,stroke-width:2px
```

# C接口回调函数类型定义 PorterDefs.h
定义了跨语言调用的回调函数类型，使得Python、C#等外部语言可以通过C接口访问 WtDtServo 模块（数据服务）的功能。
```cpp
/**
 * @brief K线数据查询回调函数类型定义
 * @param bar K线数据数组指针
 * @param count 数据条数
 * @param isLast 是否为最后一批数据
 * 
 * 用于接收查询到的K线数据。支持分批次回调，通过isLast参数判断是否为最后一批。
 * 调用方需要在回调中处理接收到的数据，可以进行存储、分析或其他处理。
 */
typedef void(PORTER_FLAG *FuncGetBarsCallback)(WTSBarStruct* bar, WtUInt32 count, bool isLast);

/**
 * @brief Tick数据查询回调函数类型定义
 * @param tick Tick数据数组指针
 * @param count 数据条数
 * @param isLast 是否为最后一批数据
 * 
 * 用于接收查询到的Tick数据。支持分批次回调，适用于大量Tick数据的流式处理。
 */
typedef void(PORTER_FLAG *FuncGetTicksCallback)(WTSTickStruct* tick, WtUInt32 count, bool isLast);

/**
 * @brief 数据计数回调函数类型定义
 * @param dataCnt 数据总条数
 * 
 * 在开始查询数据前调用，通知调用方即将查询的数据总量，便于进行进度显示或内存预分配。
 */
typedef void(PORTER_FLAG *FuncCountDataCallback)(WtUInt32 dataCnt);

/**
 * @brief 实时Tick数据回调函数类型定义
 * @param stdCode 标准化合约代码（格式：交易所.合约代码，如SSE.600000）
 * @param tick Tick数据指针
 * 
 * 用于接收实时推送的Tick数据。当订阅的合约有新Tick数据时，该回调函数会被调用。
 * 主要用于实时行情监控、策略计算等场景。
 */
typedef void(PORTER_FLAG *FuncOnTickCallback)(const char* stdCode, WTSTickStruct* tick);

/**
 * @brief 实时K线数据回调函数类型定义
 * @param stdCode 标准化合约代码（格式：交易所.合约代码，如SSE.600000）
 * @param period 周期字符串（如"m1"=1分钟, "m5"=5分钟, "d"=日线）
 * @param bar K线数据指针
 * 
 * 用于接收实时推送的K线数据。当订阅的合约有新K线数据生成时，该回调函数会被调用。
 * 主要用于实时K线监控、技术指标计算等场景。
 */
typedef void(PORTER_FLAG *FuncOnBarCallback)(const char* stdCode, const char* period, WTSBarStruct* bar);
```

# C接口函数 WtDtServo.h/cpp
定义了 WtDtServo（数据服务）模块的C接口，使得Python、C#等外部语言可以通过C接口访问 WonderTrader 的数据服务功能。

## 初始化与版本

### 初始化数据伺服器 initialize
```cpp
/**
 * @brief 初始化数据伺服器
 * @param cfgFile 配置文件路径或配置内容（取决于isFile参数）
 * @param isFile 是否为配置文件路径（true=文件路径，false=配置内容字符串）
 * @param logCfg 日志配置文件路径
 * @param cbTick 实时Tick数据回调函数（可选）
 * @param cbBar 实时K线数据回调函数（可选）
 */
void initialize(WtString cfgFile, bool isFile, WtString logCfg, FuncOnTickCallback cbTick, FuncOnBarCallback cbBar)
{
	getRunner().initialize(cfgFile, isFile, getBinDir(), logCfg, cbTick, cbBar);
}
```

### 获取版本信息 get_version

## K线数据查询

### 按时间范围查询K线 get_bars_by_range
```cpp
/**
 * @brief 按时间范围查询K线数据
 * @param stdCode 标准化合约代码（格式：交易所.合约代码，如SSE.600000）
 * @param period 周期字符串（"m1"=1分钟, "m5"=5分钟, "d"=日线等）
 * @param beginTime 开始时间（格式：yyyymmddHHMM或yyyymmdd，取决于周期）
 * @param endTime 结束时间（格式：yyyymmddHHMM或yyyymmdd，取决于周期）
 * @param cb K线数据回调函数
 * @param cbCnt 数据计数回调函数
 * @return 查询到的数据条数
 */
WtUInt32 get_bars_by_range(const char* stdCode, const char* period, WtUInt64 beginTime, WtUInt64 endTime, FuncGetBarsCallback cb, FuncCountDataCallback cbCnt)
{
	// 调用运行器查询K线数据，返回K线数据切片
	WTSKlineSlice* kData = getRunner().get_bars_by_range(stdCode, period, beginTime, endTime);
	if (kData)
	{
		uint32_t reaCnt = kData->size();
		// 调用计数回调，通知数据总条数
		cbCnt(kData->size());
		// 遍历数据切片的所有数据块
		for (std::size_t i = 0; i < kData->get_block_counts(); i++)
			// 调用数据回调，传递每个数据块的地址、大小和是否为最后一批的标识
			cb(kData->get_block_addr(i), kData->get_block_size(i), i == kData->get_block_counts() - 1);

		kData->release();
		return reaCnt;
	}
	else
	{
		return 0;
	}
}
```

### 按日期查询K线 get_bars_by_date
```cpp
/**
 * @brief 按日期查询K线数据
 * @param stdCode 标准化合约代码（格式：交易所.合约代码，如SSE.600000）
 * @param period 周期字符串（"m1"=1分钟, "m5"=5分钟, "d"=日线等）
 * @param uDate 交易日期（格式：yyyymmdd）
 * @param cb K线数据回调函数
 * @param cbCnt 数据计数回调函数
 * @return 查询到的数据条数
 */
WtUInt32 get_bars_by_date(const char* stdCode, const char* period, WtUInt32 uDate, FuncGetBarsCallback cb, FuncCountDataCallback cbCnt)
{
    // 调用运行器查询指定日期的K线数据，返回K线数据切片
	WTSKlineSlice* kData = getRunner().get_bars_by_date(stdCode, period, uDate);
	if (kData)
	{
		uint32_t reaCnt = kData->size();
        // 调用计数回调，通知数据总条数
		cbCnt(kData->size());
		for (std::size_t i = 0; i < kData->get_block_counts(); i++)
            // 调用数据回调，传递每个数据块的地址、大小和是否为最后一批的标识
			cb(kData->get_block_addr(i), kData->get_block_size(i), i == kData->get_block_counts() - 1);  

		kData->release();
		return reaCnt;
	}
	else
	{
		return 0;
	}
}
```

### 按数量查询K线 get_bars_by_count
```cpp
/**
 * @brief 按数量查询K线数据
 * @param stdCode 标准化合约代码（格式：交易所.合约代码，如SSE.600000）
 * @param period 周期字符串（"m1"=1分钟, "m5"=5分钟, "d"=日线等）
 * @param count 查询的数据条数
 * @param endTime 结束时间（格式：yyyymmddHHMM或yyyymmdd，取决于周期）
 * @param cb K线数据回调函数
 * @param cbCnt 数据计数回调函数
 * @return 查询到的数据条数
 */
WtUInt32 get_bars_by_count(const char* stdCode, const char* period, WtUInt32 count, WtUInt64 endTime, FuncGetBarsCallback cb, FuncCountDataCallback cbCnt)
{
    // 调用运行器查询指定数量的K线数据，返回K线数据切片
	WTSKlineSlice* kData = getRunner().get_bars_by_count(stdCode, period, count, endTime);  
	if (kData)
	{
		uint32_t reaCnt = kData->size();
        // 调用计数回调，通知数据总条数
		cbCnt(kData->size());        
		for(std::size_t i = 0; i< kData->get_block_counts(); i++)
            // 调用数据回调，传递每个数据块的地址、大小和是否为最后一批的标识
			cb(kData->get_block_addr(i), kData->get_block_size(i), i == kData->get_block_counts()-1);  

		kData->release();
		return reaCnt;
	}
	else
	{
		return 0;
	}
}
```

### 按日期查询秒级K线 get_sbars_by_date
```cpp
/**
 * @brief 按日期查询秒级K线数据
 * @param stdCode 标准化合约代码（格式：交易所.合约代码，如SSE.600000）
 * @param secs 秒数（如：60表示60秒K线）
 * @param uDate 交易日期（格式：yyyymmdd）
 * @param cb K线数据回调函数
 * @param cbCnt 数据计数回调函数
 * @return 查询到的数据条数
 */
WtUInt32 get_sbars_by_date(const char* stdCode, WtUInt32 secs, WtUInt32 uDate, FuncGetBarsCallback cb, FuncCountDataCallback cbCnt)
{
    // 调用运行器查询指定日期的秒级K线数据，返回K线数据切片
	WTSKlineSlice* kData = getRunner().get_sbars_by_date(stdCode, secs, uDate);  
	if (kData)
	{
		uint32_t reaCnt = kData->size();
        // 调用计数回调，通知数据总条数
		cbCnt(kData->size());                                                    
		for (std::size_t i = 0; i < kData->get_block_counts(); i++)
            // 调用数据回调，传递每个数据块的地址、大小和是否为最后一批的标识
			cb(kData->get_block_addr(i), kData->get_block_size(i), i == kData->get_block_counts() - 1);  
		kData->release();
		return reaCnt;
	}
	else
	{
		return 0;
	}
}
```

## Tick数据查询

### 按时间范围查询Tick get_ticks_by_range
```cpp
/**
 * @brief 按时间范围查询Tick数据
 * @param stdCode 标准化合约代码（格式：交易所.合约代码，如SSE.600000）
 * @param beginTime 开始时间（格式：yyyymmddHHMMSS）
 * @param endTime 结束时间（格式：yyyymmddHHMMSS）
 * @param cb Tick数据回调函数
 * @param cbCnt 数据计数回调函数
 * @return 查询到的数据条数
 */
WtUInt32	get_ticks_by_range(const char* stdCode, WtUInt64 beginTime, WtUInt64 endTime, FuncGetTicksCallback cb, FuncCountDataCallback cbCnt)
{
    // 调用运行器查询Tick数据，返回Tick数据切片
	WTSTickSlice* slice = getRunner().get_ticks_by_range(stdCode, beginTime, endTime);
	if (slice)
	{
		uint32_t reaCnt = 0;
		uint32_t blkCnt = slice->get_block_counts();
        // 调用计数回调，通知数据总条数
		cbCnt(slice->size());

		for(uint32_t sIdx = 0; sIdx < blkCnt; sIdx++)
		{
            // 调用数据回调，传递每个数据块的地址、大小和是否为最后一批的标识
			cb(slice->get_block_addr(sIdx), slice->get_block_size(sIdx), sIdx == blkCnt - 1);
			reaCnt += slice->get_block_size(sIdx);
		}
		
		slice->release();
		return reaCnt;
	}
	else
	{
		return 0;
	}
}
```

### 按日期查询Tick get_ticks_by_date
```cpp
/**
 * @brief 按日期查询Tick数据
 * @param stdCode 标准化合约代码（格式：交易所.合约代码，如SSE.600000）
 * @param uDate 交易日期（格式：yyyymmdd）
 * @param cb Tick数据回调函数
 * @param cbCnt 数据计数回调函数
 * @return 查询到的数据条数
 * 
 * 查询指定交易日的所有Tick数据，将查询结果通过回调函数返回。
 * 支持分批次回调，通过isLast参数标识是否为最后一批数据。
 */
WtUInt32 get_ticks_by_date(const char* stdCode, WtUInt32 uDate, FuncGetTicksCallback cb, FuncCountDataCallback cbCnt)
{
    // 调用运行器查询指定日期的Tick数据，返回Tick数据切片
	WTSTickSlice* slice = getRunner().get_ticks_by_date(stdCode, uDate);
	if (slice)
	{
		uint32_t reaCnt = 0;
		uint32_t blkCnt = slice->get_block_counts();
        // 调用计数回调，通知数据总条数
		cbCnt(slice->size());          

		for (uint32_t sIdx = 0; sIdx < blkCnt; sIdx++)
		{
            // 调用数据回调，传递每个数据块的地址、大小和是否为最后一批的标识
			cb(slice->get_block_addr(sIdx), slice->get_block_size(sIdx), sIdx == blkCnt - 1);
			reaCnt += slice->get_block_size(sIdx);
		}

		slice->release();
		return reaCnt;
	}
	else
	{
		return 0;
	}
}
```

### 按数量查询Tick get_ticks_by_count
```cpp
/**
 * @brief 按数量查询Tick数据
 * @param stdCode 标准化合约代码（格式：交易所.合约代码，如SSE.600000）
 * @param count 查询的数据条数
 * @param endTime 结束时间（格式：yyyymmddHHMMSS）
 * @param cb Tick数据回调函数
 * @param cbCnt 数据计数回调函数
 * @return 查询到的数据条数
 * 
 * 查询指定结束时间之前的N条Tick数据，将查询结果通过回调函数返回。
 * 如果数据不足，返回实际可用的数据条数。
 */
WtUInt32	get_ticks_by_count(const char* stdCode, WtUInt32 count, WtUInt64 endTime, FuncGetTicksCallback cb, FuncCountDataCallback cbCnt)
{
    // 调用运行器查询指定数量的Tick数据，返回Tick数据切片
	WTSTickSlice* slice = getRunner().get_ticks_by_count(stdCode, count, endTime);
	if (slice)
	{
		uint32_t reaCnt = 0;
		uint32_t blkCnt = slice->get_block_counts();
        // 调用计数回调，通知数据总条数
		cbCnt(slice->size());

		for (uint32_t sIdx = 0; sIdx < blkCnt; sIdx++)
		{
            // 调用数据回调，传递每个数据块的地址、大小和是否为最后一批的标识
			cb(slice->get_block_addr(sIdx), slice->get_block_size(sIdx), sIdx == blkCnt - 1);
			reaCnt += slice->get_block_size(sIdx);
		}

		slice->release();
		return reaCnt;
	}
	else
	{
		return 0;
	}
}
```

## 实时数据订阅

### 订阅实时Tick subscribe_tick
```cpp
/**
 * @brief 订阅实时Tick数据
 * @param stdCode 标准化合约代码（格式：交易所.合约代码，如SSE.600000）
 * @param bReplace 是否替换现有的订阅列表（true=替换，false=追加）
 */
void subscribe_tick(const char* stdCode, bool bReplace)
{
	getRunner().sub_tick(stdCode, bReplace);
}
```

### 订阅实时K线 subscribe_bar
```cpp
/**
 * @brief 订阅实时K线数据
 * @param stdCode 标准化合约代码（格式：交易所.合约代码，如SSE.600000）
 * @param period 周期字符串（"m1"=1分钟, "m5"=5分钟, "d"=日线等）
 */
void subscribe_bar(const char* stdCode, const char* period)
{
	getRunner().sub_bar(stdCode, period);
}
```

## 缓存管理

### 清理数据缓存 clear_cache
```cpp
/**
 * @brief 清理数据缓存
 */
void clear_cache()
{
	getRunner().clear_cache();
}
```

# WtDtRunner.h/cpp

## 成员
- **核心管理器**
  - `WTSBaseDataMgr _bd_mgr`：基础数据管理器（管理合约、品种等基础数据）。
  - `WTSHotMgr _hot_mgr`：主力合约管理器（管理主力合约切换规则）。
  - `WtDataStorage* _data_store`：数据存储对象指针（在构造函数中初始化为NULL）。
  - `WtDataManager _data_mgr`：数据管理器对象（管理数据查询、缓存等）。
  - `ParserAdapterMgr _parsers`：解析器适配器管理器（管理所有解析器适配器）。
- **状态与配置**
  - `bool _is_inited`：初始化标志：标识是否已初始化。
- **回调函数**
  - `FuncOnTickCallback _cb_tick`：实时Tick数据回调函数指针。
  - `FuncOnBarCallback _cb_bar`：实时KLine数据回调函数指针。
- **订阅管理**
  - `StraSubMap _tick_sub_map`：Tick数据订阅表，存储外部订阅的Tick数据（键=合约代码）。
  - `StraSubMap _tick_innersub_map`：Tick数据内部订阅表，存储内部订阅的Tick数据（键=合约代码）。
    - typedef wt_hashmap\<std::string, `SubFlags`\> StraSubMap：订阅映射表类型定义（键=合约代码，值=订阅标志集合）。
    - typedef std::set\<uint32\_t\> `SubFlags`：订阅标志集合类型定义（用于标识订阅状态）。
      - 0：原始数据（不复权）
      - 1：前复权
      - 2：后复权
  - `StdUniqueMutex _mtx_subs`：订阅表互斥锁：保护多线程访问订阅表。
  - `StdUniqueMutex _mtx_innersubs`：内部订阅表互斥锁：保护多线程访问内部订阅表。

## 方法

### 初始化与启动

#### 初始化数据服务 initialize

#### 启动所有解析器 start
```cpp
void WtDtRunner::start()
{
    // 调用解析器管理器的run()方法，启动所有解析器
	_parsers.run();
}
```

### 实时数据处理

#### 触发Tick回调 trigger_tick

#### 触发K线回调 trigger_bar

#### 处理Tick数据 proc_tick

### 数据订阅

#### 订阅Tick数据 sub_tick

#### 订阅K线数据 sub_bar

### 数据查询接口

#### 按时间范围查询K线 get_bars_by_range

#### 按日期查询K线 get_bars_by_date

#### 按时间范围查询Tick get_ticks_by_range

#### 按数量查询K线 get_bars_by_count

#### 按数量查询Tick get_ticks_by_count

#### 按日期查询Tick get_ticks_by_date

#### 按日期查询秒级K线 get_sbars_by_date

# WtDataManager.h/cpp

## 成员
- **核心管理器指针**
  - `IRdmDtReader* _reader`：随机数据读取器指针（用于读取历史数据）。
  - `IBaseDataMgr* _bd_mgr`：基础数据管理器指针（用于获取合约信息等）。
  - `IHotMgr* _hot_mgr`：主力合约管理器指针（用于处理主力合约切换）。
  - `WtDtRunner* _runner`：数据服务运行器指针（用于触发回调等）。
- **函数指针**
  - `FuncDeleteRdmDtReader _remover`：数据读取器删除函数指针（用于动态库创建的读取器）。
- **配置与状态**
  - `bool _align_by_section`：是否按交易时段对齐K线数据。
- **K线缓存**
  - `typedef struct _BarCache`：K线缓存结构。
    - `WTSKlineData* _bars`：缓存的K线数据指针。
    - `uint64_t _last_bartime`：最后一条K线的时间戳。
    - `WTSKlinePeriod _period`：K线周期。
    - `uint32_t _times`：周期倍数。
  - `typedef wt_hashmap<std::string, BarCache> BarCacheMap`：K线缓存映射表类型定义（键=合约代码+日期+周期）。
  - `BarCacheMap _bars_cache`：K线数据缓存映射表：存储已查询的K线数据。
- **实时K线缓存**
  - `typedef WTSHashMap<std::string> RtBarMap`：实时K线映射表类型定义（键=合约代码+周期）。
  - `RtBarMap* _rt_bars`：实时K线映射表指针：存储实时生成的K线数据。
  - `StdUniqueMutex _mtx_rtbars`：实时K线映射表互斥锁：保护多线程访问。

## 方法

### 初始化

#### 初始化 init

#### 初始化数据存储模块 initStore

### IRdmDtReaderSink 接口实现

#### 获取基础数据管理接口指针 get_basedata_mgr

#### 获取主力切换规则管理接口指针 get_hot_mgr

#### 输出数据读取模块的日志 reader_log

### Level-2数据查询接口

#### 查询委托队列数据切片 get_order_queue_slice

#### 查询逐笔委托数据切片 get_order_detail_slice

#### 查询逐笔成交数据切片 get_transaction_slice

### K线与Tick数据查询接口

#### 按日期查询Tick数据切片 get_tick_slice_by_date

#### 按日期查询秒级K线数据切片 get_skline_slice_by_date

#### 按日期查询K线数据切片 get_kline_slice_by_date

#### 按时间范围查询Tick数据切片 get_tick_slices_by_range

#### 按时间范围查询K线数据切片 get_kline_slice_by_range

#### 按数量查询Tick数据切片 get_tick_slice_by_count

#### 按数量查询K线数据切片 get_kline_slice_by_count

### 复权因子

#### 获取复权因子 get_exright_factor

### 实时K线管理

#### 订阅实时K线数据 subscribe_bar

#### 清除所有订阅的K线 clear_subbed_bars

#### 更新K线数据 update_bars

# ParserAdapter.h/cpp

## 行情解析器适配器 ParserAdapter
```cpp
class ParserAdapter : public IParserSpi, private boost::noncopyable 
```
参考 [Includes/note.ipynb/API 接口层/行情解析 IParserApi.h/行情解析器回调接口 IParserSpi](../Includes/note.ipynb)

### 成员
- **核心接口指针**
  - `IParserApi* _parser_api`：解析器API接口指针。
  - `WTSBaseDataMgr* _bd_mgr`：基础数据管理器指针。
  - `WtDtRunner* _dt_runner`：数据服务运行器指针。
- **函数指针**
  - `FuncDeleteParser _remover`：解析器删除函数指针（用于动态库创建的解析器）。
- **状态与配置**
  - `bool _stopped`：停止标志：标识解析器是否已停止。
  - `WTSVariant* _cfg`：配置信息指针（用于初始化）。
  - `std::string _id`：解析器ID（唯一标识）。
- **过滤器**
  - `ExchgFilter _exchg_filter`：交易所过滤器：只处理指定交易所的数据。
  - `ExchgFilter _code_filter`：合约代码过滤器：只处理指定合约的数据。
    - typedef wt_hashset\<std::string\> ExchgFilter：交易所过滤器类型定义（字符串集合）。

### 方法

#### 初始化与生命周期

##### 初始化 (从配置) init

##### 初始化 (外部API) initExt
```cpp
/**
 * @brief 初始化适配器（外部API）
 * @param id 解析器ID（唯一标识）
 * @param api 解析器API指针（外部创建）
 * @return 是否初始化成功
 */
bool ParserAdapter::initExt(const char* id, IParserApi* api)
```

##### 释放适配器资源 release

##### 启动解析器 run

#### IParserSpi 接口实现

##### 处理合约列表回调 handleSymbolList

##### 处理行情数据回调 handleQuote

##### 处理委托队列数据回调 handleOrderQueue

##### 处理逐笔成交数据回调 handleTransaction

##### 处理逐笔委托数据回调 handleOrderDetail

##### 处理解析器日志回调 handleParserLog

## 行情解析器管理器 ParserAdapterMgr

### 成员
- `ParserAdapterMap _adapters`：适配器映射表，存储所有解析器适配器（ID->适配器指针）
  - typedef wt_hashmap\<std::string, `ParserAdapterPtr`\>	ParserAdapterMap：解析器适配器映射表类型定义（ID->适配器）
  - typedef std::shared_ptr\<`ParserAdapter`\> ParserAdapterPtr：解析器适配器智能指针类型定义

### 启动所有适配器 run
```cpp
/**
 * @brief 启动所有适配器
 */
void ParserAdapterMgr::run()
{
	for (auto it = _adapters.begin(); it != _adapters.end(); it++)
	{
		it->second->run();
	}

	WTSLogger::info("{} parsers started", _adapters.size());
}
```

# WtHelper.h/cpp

## 成员
`static std::string _bin_dir`：存储二进制模块目录路径

## 方法

### 获取当前工作目录 get_cwd
```cpp
/**
 * @brief 获取当前工作目录的实现
 * @return 当前工作目录的字符串指针（标准化路径格式）
 */
const char* WtHelper::get_cwd()
{
	static std::string _cwd;		// 静态局部变量：缓存工作目录路径，确保只查询一次
	if(_cwd.empty())
	{
		char   buffer[255];
#ifdef _MSC_VER 					// 如果是Windows平台
		_getcwd(buffer, 255);
#else   							// 如果是Unix/Linux平台
		getcwd(buffer, 255);
#endif
		_cwd = buffer;
		_cwd = StrUtil::standardisePath(_cwd);
	}
	return _cwd.c_str();
}
```

### 获取模块目录路径 get_module_dir
```cpp
static const char* get_module_dir(){ return _bin_dir.c_str(); }
```

### 设置模块目录路径 set_module_dir
```cpp
static void set_module_dir(const char* mod_dir){ _bin_dir = mod_dir; }
```